# Detecting cyberattacks in a healthcare monitoring system

## Dataset: WUSTL-EHMS-2020

| Field | Description |
|---|---|
| **Source** | [Kaggle: aghubaish/wustl-ehms-2020-dataset](https://www.kaggle.com/datasets/aghubaish/wustl-ehms-2020-dataset), uploaded by co-author Ali Ghubaish (Washington University in St. Louis) |
| **Paper** | Hady et al., *Intrusion Detection System for Healthcare Systems Using Medical and Network Data: A Comparison Study*, IEEE Access, 2020 |
| **Size** | 16,318 samples, 43 features |
| **One row** | One network flow (a captured communication between devices in the monitoring system), together with the patient's vital-sign readings transmitted at that moment |
| **Prediction** | Whether a network flow is part of a cyberattack |
| **Target** | `Label`: 0 = Normal, 1 = Attack (the attack class is the minority, about 12.5% of rows) |

The data comes from a real-time Enhanced Healthcare Monitoring System (EHMS) testbed. Sensors on a patient send vital signs over a network to a monitoring server, while an attacker performs man-in-the-middle attacks (spoofing and data injection).

The 43 features are:
- **35 network flow metrics:** packets, bytes, rates, jitter, duration, and so on.
- **8 patient biometric values:** temperature, SpO2, pulse rate, systolic/diastolic blood pressure, heart rate, respiration rate and ST segment.

**Synthetic dataset:** `synthetic_signal.csv` is only loaded for a quick pipeline test (`is_pipeline_test = True` in the data acquisition cell). It is not needed for the real run. To create it, run `synthetic_data_generator.py` from this folder.

## Objective

Train and compare three classification methods, **KNN**, **SVM** and **Random Forest**, on the full dataset, and find the best one for detecting attacks.

### Method

- **Split:** stratified 70/30 train/test split (`random_state=42`). The test set is used for reporting only, never for fitting or tuning.
- **Validation:** 5-fold stratified cross-validation on the training set. Hyperparameters and decision thresholds are tuned on this same data, so the CV scores are slightly optimistic. The test-set results are the unbiased estimate.
- **Preprocessing:** encoding and scaling are done inside scikit-learn `Pipeline`s, so they are fitted on the training folds only.
- **Metrics:** precision, recall, specificity (false alarms), F1 and ROC-AUC for the Attack class. Accuracy would be misleading, because only about 12.5% of the flows are attacks.

### How the work developed

1. **v1, default configuration.** A grid search over each model's main hyperparameters (chosen by F1), standard preprocessing (balanced class weights for the SVM and Random Forest) and the default decision threshold. This is the first result.
2. **Aiming for 90% recall (v2 and v3).** v1 missed 24% to 34% of the attacks. For a medical monitoring device a missed attack is worse than a false alarm, so I tried to catch at least 90% of the attacks with as few false alarms as possible. The 90% was not the goal at the start and it is not a clinical standard: it is a level chosen after seeing v1. v2 and v3 are two tuning attempts to reach it.
3. **Study case.** The last cell moves the threshold to other recall levels (85%, 95% and 98%) to show what higher recall costs in false alarms. A recall close to 100% is not realistic and is not a goal.

### Notebook structure

1. Data acquisition and exploratory data analysis (EDA)
2. Preprocessing: drop constant columns and identifier/leakage columns
3. Train/test split
4. **v1, default configuration:** grid search for each model, then its test evaluation
5. **v2, log transform + threshold tuning for the recall goal:** ablation of `log1p` on the network features, then a decision threshold tuned for the goal, then its test evaluation
6. **v3, wider grids:** extended hyperparameter search with the same goal, then its test evaluation
7. Comparison of v1, v2 and v3 on the test set
8. Study case: all versions at recall levels of 85%, 90%, 95% and 98%

In [ ]:
# Install kagglehub if not already installed
try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

In [ ]:
####################
# Data acquisition #
####################

import os
import pandas as pd

is_pipeline_test = False

if(is_pipeline_test):
  print("Loading synthetic dataset: synthetic_signal.csv")
  df = pd.read_csv("synthetic_signal.csv")
  print(f"Successfully loaded synthetic_signal.csv with shape {df.shape}")
  display(df.head())
else:
  print("Loading real dataset aghubaish/wustl-ehms-2020-dataset")
  path = kagglehub.dataset_download("aghubaish/wustl-ehms-2020-dataset")
  print("Path to dataset files:", path)

  # Find the CSV file inside the downloaded path
  files = os.listdir(path)
  csv_files = [f for f in files if f.endswith('.csv')]
  print("Downloaded files:", files)

  # Load the CSV file into a DataFrame if found
  if csv_files:
      csv_path = os.path.join(path, csv_files[0])
      df = pd.read_csv(csv_path)
      print(f"Successfully loaded {csv_files[0]} with shape {df.shape}")
      display(df.head())
  else:
      raise Exception("No CSV file found in the downloaded dataset path.")

In [ ]:
###################################
# Exploratory data analysis (EDA) #
###################################

print("=== Dataset Info (Fields & Data Types) ===")
print(df.info())

print("\n=== Missing Values ===")
missing_vals = df.isnull().sum()
print(missing_vals[missing_vals > 0] if missing_vals.sum() > 0 else "No missing values found.")

print("\n=== Duplicate Rows ===")
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(exclude=['number']).columns.tolist()
print(f"\n=== Numerical Fields ({len(numerical_cols)}) ===\n{numerical_cols}")
print(f"\n=== Categorical Fields ({len(categorical_cols)}) ===\n{categorical_cols}")

# constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
constant_cols = ['Dir', 'SrcAddr', 'DstAddr', 'Dport', 'SrcGap', 'DstGap',
                 'DIntPktAct', 'dMinPktSz', 'Trans', 'DstMac']
print(f"\n=== Constant Fields (Single Unique Value) ===\n{constant_cols}")

# potential_identifiers = [col for col in df.columns if df[col].nunique() == len(df)]
potential_identifiers = ['SrcMac', 'Attack Category', 'Sport', 'Packet_num']
print(f"\n=== Potential Unique Identifiers ===\n{potential_identifiers}")

# balance between the two target classes
target_column = 'Label'  # Typically 'Label' in WUSTL-EHMS dataset
if target_column in df.columns:
    print(f"\n=== Class Balance for '{target_column}' ===")
    class_counts = df[target_column].value_counts()
    class_percentages = df[target_column].value_counts(normalize=True) * 100
    for val, count in class_counts.items():
        print(f"Class {val}: {count} ({class_percentages[val]:.2f}%)")
else:
    print(f"\nTarget column '{target_column}' not found. Let's look for columns with 'attack' or 'label' in name:")
    possible_targets = [col for col in df.columns if 'label' in col.lower() or 'attack' in col.lower()]
    print(possible_targets)

In [ ]:
#################
# Preprocessing #
#################

# Cleanup
print(f"Dropping constant columns: {constant_cols}")
df_cleaned = df.drop(columns=constant_cols, errors='ignore')

print(f"Dropping identifiers columns: {potential_identifiers}")
df_cleaned = df_cleaned.drop(columns=potential_identifiers, errors='ignore')

# Duplicate rows would leak identical samples into both train and test
n_before = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates()
print(f"Duplicate rows removed after dropping columns: {n_before - len(df_cleaned)}")

display(df_cleaned.head())

In [ ]:
####################
# Train/test split #
####################

from sklearn.model_selection import train_test_split

# Separate features and target from df_cleaned
features = df_cleaned.drop(columns=['Label'])
target = df_cleaned['Label']

# Safety check: all features must be numeric before scaling
non_numeric = [c for c in features.select_dtypes(exclude='number').columns if c != 'Flgs']
assert not non_numeric, f"Unexpected non-numeric columns: {non_numeric}"

# Perform a stratified 70/30 training-test split
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.30, random_state=42, stratify=target
)

print("\n=== Splitting Summary ===")
print(f"Training set shape: {features_train.shape} (target: {target_train.shape})")
print(f"Test set shape (not used for fitting): {features_test.shape} (target: {target_test.shape})")
print(f"\nTraining class distribution:\n{target_train.value_counts(normalize=True).to_string()}")
print(f"\nTest class distribution:\n{target_test.value_counts(normalize=True).to_string()}")

In [ ]:
###################################
# Model training (5-fold CV) - V1 #
###################################

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

categorical_features = [col for col in categorical_cols if col in features_train.columns]
numeric_features = [col for col in numerical_cols if col in features_train.columns]

print(f"Categorical Features: {categorical_features}")
print(f"Numeric Features (scaling only): {numeric_features}\n")

# Baseline preprocessor: StandardScaler for numeric features, OneHotEncoder for categoricals (no log transform)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fitted_searches = {}

# Metrics computed during CV; refit optimizes F1
scoring_metrics = {
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall',
    'roc_auc': 'roc_auc'
}

def print_best_cv_results(name, search):
    best_idx = search.best_index_
    print(f"=== {name} Best Configuration CV Performance ===")
    print(f"Best Params: {search.best_params_}")
    for metric in scoring_metrics.keys():
        mean_val = search.cv_results_[f'mean_test_{metric}'][best_idx]
        std_val = search.cv_results_[f'std_test_{metric}'][best_idx]
        print(f"{metric.upper():9}: {mean_val:.4f} ± {std_val:.4f}")
    print()

models = {
    'KNN': (
        KNeighborsClassifier(),
        {'model__n_neighbors': [1, 2, 3, 5, 7, 11, 15, 21],
         'model__weights': ['uniform', 'distance']}
    ),
    'SVM': (
        SVC(class_weight='balanced', random_state=42),
        {'model__C': [0.1, 1, 10, 100],
         'model__gamma': ['scale', 0.01, 0.1, 1]}
    ),
    'RandomForest': (
        RandomForestClassifier(class_weight='balanced', random_state=42),
        {'model__n_estimators': [100, 300],
         'model__max_depth': [None, 10, 20],
         'model__min_samples_leaf': [1, 5, 10]}
    ),
}

for name, (estimator, param_grid) in models.items():
    print(f"--- Starting Grid Search for {name} ---")
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', estimator)])
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring_metrics,
        refit='f1',
        cv=cv_strategy,
        n_jobs=-1
    )
    search.fit(features_train, target_train)
    fitted_searches[name] = search
    print_best_cv_results(name, search)

# === Summary table ===
summary_data = []
for model_name, search in fitted_searches.items():
    best_idx = search.best_index_
    row = {'Model': model_name, 'Best Params': str(search.best_params_)}
    for metric in scoring_metrics.keys():
        row[f'mean_test_{metric}'] = search.cv_results_[f'mean_test_{metric}'][best_idx]

In [ ]:
###################################
# Evaluation on the test set - V1 #
###################################

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve,
    precision_recall_fscore_support, roc_auc_score
)

# The test set is used for reporting only: design decisions come from CV.
# Each version is evaluated once, right after its training cell. Results are
# collected in test_results and compared in the last cell.
if not isinstance(globals().get('test_results'), dict):
    test_results = {}

# Chart style: one fixed colour per classifier (colour follows the model, in the
# same order everywhere), thin lines, recessive grid.
MODEL_COLORS = {'KNN': '#2a78d6', 'SVM': '#eb6834', 'RandomForest': '#1baf7a'}
SURFACE, INK, INK_2, MUTED, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
CM_CMAP = LinearSegmentedColormap.from_list('blue_seq', ['#cde2fb', '#104281'])

def style_axes(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(MUTED)
    ax.tick_params(colors=INK_2)

def plot_roc(version, curves, rows):
    """ROC curves of all classifiers on one axes. Dots mark the threshold used."""
    fig, ax = plt.subplots(figsize=(5.6, 5.2), facecolor=SURFACE)
    style_axes(ax)
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1, color=MUTED, label='Chance (AUC = 0.500)')
    for row in rows:
        name = row['Model']
        scores, preds = curves[name]
        fpr, tpr, _ = roc_curve(target_test, scores)
        ax.plot(fpr, tpr, linewidth=2, color=MODEL_COLORS[name],
                label=f"{name} (AUC = {row['ROC-AUC']:.3f})")
        # Operating point: where the threshold used in the report lands on the curve
        tn, fp, fn, tp = confusion_matrix(target_test, preds).ravel()
        ax.scatter([fp / (fp + tn)], [tp / (tp + fn)], s=70, color=MODEL_COLORS[name],
                   edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel('False positive rate', color=INK_2)
    ax.set_ylabel('True positive rate (recall)', color=INK_2)
    ax.set_title(f'ROC curves on the test set: {version}', color=INK, loc='left', fontsize=11)
    ax.text(0.0, -0.17, 'Dots mark the threshold used in the classification report.',
            transform=ax.transAxes, color=INK_2, fontsize=8.5)
    ax.legend(loc='lower right', frameon=False, labelcolor=INK, fontsize=9)
    plt.tight_layout()
    plt.show()

def plot_confusion(version, curves):
    """Confusion matrices side by side. Colour = share of each actual class."""
    fig, axes = plt.subplots(1, len(curves), figsize=(3.4 * len(curves), 3.5), facecolor=SURFACE)
    axes = np.atleast_1d(axes)
    for ax, (name, (scores, preds)) in zip(axes, curves.items()):
        cm = confusion_matrix(target_test, preds)
        share = cm / cm.sum(axis=1, keepdims=True)
        ax.imshow(share, cmap=CM_CMAP, vmin=0, vmax=1)
        for r in range(2):
            for c in range(2):
                ax.text(c, r, f"{cm[r, c]}\n({share[r, c]:.0%})", ha='center', va='center',
                        color='white' if share[r, c] > 0.5 else INK, fontsize=10)
        ax.set_xticks([0, 1], ['Normal', 'Attack'], color=INK_2)
        ax.set_yticks([0, 1], ['Normal', 'Attack'], color=INK_2)
        ax.set_xlabel('Predicted', color=INK_2)
        ax.set_ylabel('Actual', color=INK_2)
        ax.set_title(name, color=INK, fontsize=10)
        for spine in ax.spines.values():
            spine.set_visible(False)
    fig.suptitle(f'Confusion matrices on the test set: {version} (colour = share of each actual class)',
                 color=INK, x=0.01, ha='left', fontsize=10.5)
    plt.tight_layout()
    plt.show()

def evaluate_on_test(version, version_models, version_thresholds=None):
    """Score fitted models on the test set, print reports, store one row per model and plot.

    version_thresholds=None means an untuned model: predict() is used, which
    corresponds to the default threshold (0.5 for probabilities, 0.0 for SVM
    decision-function scores). Otherwise the tuned threshold is applied.
    """
    rows = []
    curves = {}
    for name, model in version_models.items():
        print("=========================================================")
        print(f" {version} - {name}: Test Set Evaluation")
        print("=========================================================")

        # Get scores/probabilities on the test partition
        if hasattr(model, "predict_proba"):
            test_scores = model.predict_proba(features_test)[:, 1]
            threshold = 0.5
        else:
            test_scores = model.decision_function(features_test)
            threshold = 0.0

        if version_thresholds is None:
            # Untuned model: use predict() (ties at exactly 0.5, e.g. KNN with
            # even k, go to class 0)
            preds = model.predict(features_test)
        else:
            threshold = version_thresholds[name]
            preds = (test_scores >= threshold).astype(int)

        print(f"\n--- [Threshold: {threshold:.4f}] Classification Report ---")
        print(classification_report(target_test, preds, target_names=['Normal (0)', 'Attack (1)']))
        print("Confusion Matrix:")
        print(confusion_matrix(target_test, preds))
        print()

        # Metrics for the positive/Attack class
        prec, rec, f1, _ = precision_recall_fscore_support(
            target_test, preds, average='binary', pos_label=1
        )
        tn, fp, fn, tp = confusion_matrix(target_test, preds).ravel()
        rows.append({
            'Version': version,
            'Model': name,
            'Threshold': threshold,
            'Precision (Attack)': prec,
            'Recall (Attack)': rec,
            'Specificity': tn / (tn + fp),
            'False alarms': int(fp),
            'F1-Score (Attack)': f1,
            'ROC-AUC': roc_auc_score(target_test, test_scores)
        })
        curves[name] = (test_scores, preds)

    test_results[version] = rows
    print(f"Test results for {version}:")
    display(pd.DataFrame(rows).round(4))
    plot_roc(version, curves, rows)
    plot_confusion(version, curves)

# v1: baseline grid search (training cell), default threshold
evaluate_on_test(
    'v1 (baseline)',
    {name: search.best_estimator_ for name, search in fitted_searches.items()}
)

In [ ]:
##########################################################
# Ablation Study & Threshold Tuning on Training Set - V2 #
##########################################################

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.model_selection import GridSearchCV, cross_validate, TunedThresholdClassifierCV
from sklearn.metrics import make_scorer, roc_curve, confusion_matrix, recall_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Reuses from earlier cells: cv_strategy, preprocessor (baseline), numeric_features,
# categorical_features, features_train, target_train

# --- Objective: catch at least RECALL_TARGET of the attacks, with as few false alarms as possible ---
# In a medical monitoring setting a missed attack is worse than a false alarm. The 90% level was chosen
# after seeing v1 (it is not a clinical standard). Change the target here; v3 reuses it.
RECALL_TARGET = 0.90

def specificity_at_recall(y_true, y_score):
    """Best specificity (1 - false-alarm rate) over all thresholds that reach RECALL_TARGET.
    Works on continuous scores, so it can rank hyperparameters without fixing a threshold."""
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(1 - fpr[tpr >= RECALL_TARGET].min())

def specificity_if_recall_met(y_true, y_pred):
    """Hard-label version for choosing the threshold: specificity if recall reaches the target,
    otherwise a penalty that grows with the recall shortfall."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn)
    return tn / (tn + fp) if recall >= RECALL_TARGET else recall - 1.0

# Hyperparameters are ranked on scores (no fixed threshold); the threshold is then tuned on the winner.
ranking_scorer = make_scorer(specificity_at_recall, response_method=('decision_function', 'predict_proba'))
threshold_scorer = make_scorer(specificity_if_recall_met)
search_scoring = {'spec_at_recall': ranking_scorer, 'roc_auc': 'roc_auc'}
nested_scoring = {
    'recall': 'recall',
    'specificity': make_scorer(recall_score, pos_label=0),
    'precision': 'precision',
}

# --- Feature groups for the log transform ---
vital_signs = ['Temp', 'SpO2', 'Pulse_Rate', 'SYS', 'DIA', 'Heart_rate', 'Resp_Rate', 'ST']
network_features = [c for c in numeric_features if c not in vital_signs]
vitals_to_scale = [c for c in numeric_features if c in vital_signs]

# Safety check: log1p requires non-negative values
negatives = (features_train[network_features] < 0).sum()
assert negatives.sum() == 0, f"Negative values found:\n{negatives[negatives > 0]}"
print("All network features are non-negative: log1p is safe.\n")

# Preprocessor with log1p + scaling for network features, scaling only for vitals, one-hot for Flgs
log1p_preprocessor = ColumnTransformer(
    transformers=[
        ('net', Pipeline(steps=[
            ('log1p', FunctionTransformer(np.log1p)),
            ('scaler', StandardScaler())
        ]), network_features),
        ('vital', StandardScaler(), vitals_to_scale),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)
preprocessors = {'standard': preprocessor, 'log1p': log1p_preprocessor}
# Trees are invariant to monotonic transforms, so log1p cannot help Random Forest
log1p_skipped = {'RandomForest'}

# Same models and grids as in the v1 training cell
models = {
    'KNN': (
        KNeighborsClassifier(),
        {'model__n_neighbors': [1, 2, 3, 5, 7, 11, 15, 21],
         'model__weights': ['uniform', 'distance']}
    ),
    'SVM': (
        SVC(class_weight='balanced', random_state=42),
        {'model__C': [0.1, 1, 10, 100],
         'model__gamma': ['scale', 0.01, 0.1, 1]}
    ),
    'RandomForest': (
        RandomForestClassifier(class_weight='balanced', random_state=42),
        {'model__n_estimators': [100, 300],
         'model__max_depth': [None, 10, 20],
         'model__min_samples_leaf': [1, 5, 10]}
    ),
}

def fmt(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

v2_preprocessing = {}   # preprocessing chosen per model, reused by v3
final_models = {}
final_thresholds = {}
ablation_records = []
configuration_names = {'standard': '1. Baseline preprocessing', 'log1p': '2. + Log transform'}

for model_name, (estimator, param_grid) in models.items():
    print("=========================================")
    print(f" Ablation Study for: {model_name}")
    print("=========================================")

    # --- Configs 1 and 2: baseline vs log1p preprocessing, ranked by specificity at the recall target ---
    searches = {}
    for pre_name in ('standard', 'log1p'):
        if pre_name == 'log1p' and model_name in log1p_skipped:
            print("--- log1p skipped: trees are invariant to monotonic transforms ---")
            continue
        print(f"--- Grid search with {pre_name} preprocessing ---")
        pipeline = Pipeline(steps=[('preprocessor', preprocessors[pre_name]), ('model', estimator)])
        search = GridSearchCV(
            pipeline, param_grid, scoring=search_scoring, refit='spec_at_recall',
            cv=cv_strategy, n_jobs=-1
        )
        search.fit(features_train, target_train)
        searches[pre_name] = search
        i, r = search.best_index_, search.cv_results_
        ablation_records.append({
            'Model': model_name,
            'Configuration': configuration_names[pre_name],
            'Best Params': str(search.best_params_),
            'Threshold': 'n/a',
            'Spec @ recall target': fmt(r['mean_test_spec_at_recall'][i], r['std_test_spec_at_recall'][i]),
            'Recall': '-', 'Specificity': '-', 'Precision': '-',
            'ROC-AUC': fmt(r['mean_test_roc_auc'][i], r['std_test_roc_auc'][i]),
        })

    best_pre = max(searches, key=lambda k: searches[k].best_score_)
    best_search = searches[best_pre]
    v2_preprocessing[model_name] = best_pre

    # --- Config 3: + tuned threshold on the better preprocessing ---
    print(f"--- 3. + Tuned threshold on {best_pre} preprocessing (threshold chosen inside CV folds) ---")
    tuned_clf = TunedThresholdClassifierCV(best_search.best_estimator_, scoring=threshold_scorer, cv=cv_strategy)

    # 5-fold CV of the tuned model: the threshold is chosen inside each training fold.
    # Note: the hyperparameters were already selected on the whole training set, so these
    # CV scores are slightly optimistic. The test set gives the unbiased estimate.
    nested = cross_validate(
        tuned_clf, features_train, target_train,
        cv=cv_strategy, scoring=nested_scoring, n_jobs=-1
    )

    # Final tuned model: fit on the full training set to get the final threshold
    tuned_clf.fit(features_train, target_train)
    final_models[model_name] = tuned_clf
    final_thresholds[model_name] = tuned_clf.best_threshold_

    i, r = best_search.best_index_, best_search.cv_results_
    ablation_records.append({
        'Model': model_name,
        'Configuration': '3. + Tuned threshold',
        'Best Params': str(best_search.best_params_),
        'Threshold': f"{tuned_clf.best_threshold_:.4f}",
        'Spec @ recall target': '-',
        'Recall': fmt(nested['test_recall'].mean(), nested['test_recall'].std()),
        'Specificity': fmt(nested['test_specificity'].mean(), nested['test_specificity'].std()),
        'Precision': fmt(nested['test_precision'].mean(), nested['test_precision'].std()),
        'ROC-AUC': fmt(r['mean_test_roc_auc'][i], r['std_test_roc_auc'][i]),
    })
    print(f"-> Selected for {model_name}: {best_pre} preprocessing + tuned threshold "
          f"(CV recall {nested['test_recall'].mean():.3f}, specificity {nested['test_specificity'].mean():.3f})\n")

# === Summary table ===
df_ablation = pd.DataFrame(ablation_records)
print("=========================================================================")
print(f"      ABLATION STUDY RESULTS (5-fold CV, recall target {RECALL_TARGET:.0%}, optimistic)")
print("=========================================================================")
with pd.option_context('display.max_colwidth', None, 'display.width', 250):
    display(df_ablation)

In [ ]:
###################################
# Evaluation on the test set - V2 #
###################################

evaluate_on_test('v2 (log + threshold)', final_models, final_thresholds)

In [ ]:
###########################################################
# Extended Hyperparameter Tuning & Threshold Tuning  - V3 #
###########################################################

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_validate, TunedThresholdClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Reuses from the v2 cell: RECALL_TARGET, search_scoring, threshold_scorer, nested_scoring,
# preprocessors, v2_preprocessing (the preprocessing v2 selected for each model), fmt, cv_strategy
# v3 = v2 + wider hyperparameter grids. Same objective: catch at least RECALL_TARGET of the
# attacks with as few false alarms as possible.

# Define the hyperparameter configurations
models_config = {
    'KNN': {
        'estimator': KNeighborsClassifier(),
        # Larger k gives smoother scores, which a recall-oriented threshold needs (k = 1 or 2 gives
        # only a few distinct scores)
        'grid': {
            'model__n_neighbors': [3, 5, 7, 11, 15, 21, 31, 51],
            'model__weights': ['uniform', 'distance'],
            'model__metric': ['euclidean', 'manhattan']
        }
    },
    'SVM': {
        'estimator': SVC(class_weight='balanced', random_state=42),
        'grid': {
            'model__C': [1, 10, 100, 300, 1000],
            'model__gamma': ['scale', 0.03, 0.1, 0.3]
        }
    },
    'RandomForest': {
        'estimator': RandomForestClassifier(random_state=42),
        'grid': {
            'model__n_estimators': [100, 300],
            'model__max_depth': [None, 20],
            'model__min_samples_leaf': [1, 3, 5, 10],
            'model__max_features': ['sqrt', 0.3, None],
            'model__class_weight': ['balanced', 'balanced_subsample']
        }
    }
}

final_models_ext = {}
final_thresholds_ext = {}
ext_summary_records = []

for name, cfg in models_config.items():
    print(f"=========================================")
    print(f" Running v3 Pipeline & Grid Search: {name}")
    print(f"=========================================")

    # Preprocessing is the one v2 selected for this model
    pre_name = v2_preprocessing[name]
    pipeline = Pipeline(steps=[('preprocessor', preprocessors[pre_name]), ('model', cfg['estimator'])])

    # Optimize hyperparameters using CV on the training set (ranked by specificity at the recall target)
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=cfg['grid'],
        scoring=search_scoring,
        refit='spec_at_recall',
        cv=cv_strategy,
        n_jobs=-1
    )
    grid_search.fit(features_train, target_train)

    best_est = grid_search.best_estimator_
    best_params = grid_search.best_params_
    i, r = grid_search.best_index_, grid_search.cv_results_

    # Tune the decision threshold for the recall target
    tuned_clf = TunedThresholdClassifierCV(best_est, scoring=threshold_scorer, cv=cv_strategy)

    # 5-fold CV of the tuned model (threshold chosen inside each training fold).
    # Note: the hyperparameters were already selected on the whole training set, so these
    # CV scores are slightly optimistic. The test set gives the unbiased estimate.
    nested = cross_validate(
        tuned_clf, features_train, target_train,
        cv=cv_strategy, scoring=nested_scoring, n_jobs=-1
    )

    # Fit on full training set to extract final operational threshold
    tuned_clf.fit(features_train, target_train)

    final_models_ext[name] = tuned_clf
    final_thresholds_ext[name] = tuned_clf.best_threshold_

    print(f"Preprocessing: {pre_name} (selected in v2)")
    print(f"Best Params: {best_params}")
    print(f"Final Best Threshold: {tuned_clf.best_threshold_:.4f}")
    print(f"CV Recall:      {fmt(nested['test_recall'].mean(), nested['test_recall'].std())}")
    print(f"CV Specificity: {fmt(nested['test_specificity'].mean(), nested['test_specificity'].std())}")
    print(f"CV Precision:   {fmt(nested['test_precision'].mean(), nested['test_precision'].std())}")
    print(f"CV ROC-AUC:     {fmt(r['mean_test_roc_auc'][i], r['std_test_roc_auc'][i])}\n")

    ext_summary_records.append({
        'Model': name,
        'Preprocessing': pre_name,
        'Best Params': str(best_params),
        'Threshold': f"{tuned_clf.best_threshold_:.4f}",
        'Recall': fmt(nested['test_recall'].mean(), nested['test_recall'].std()),
        'Specificity': fmt(nested['test_specificity'].mean(), nested['test_specificity'].std()),
        'Precision': fmt(nested['test_precision'].mean(), nested['test_precision'].std()),
        'ROC-AUC': fmt(r['mean_test_roc_auc'][i], r['std_test_roc_auc'][i]),
    })

# Construct summary table
df_ext_summary = pd.DataFrame(ext_summary_records)
print("=========================================================================")
print(f"   SUMMARY OF EXTENDED TUNING RESULTS (5-fold CV, recall target {RECALL_TARGET:.0%}, optimistic)")
print("=========================================================================")
with pd.option_context('display.max_colwidth', None, 'display.width', 250):
    display(df_ext_summary)

In [ ]:
###################################
# Evaluation on the test set - V3 #
###################################

evaluate_on_test('v3 (wider grids)', final_models_ext, final_thresholds_ext)

In [ ]:
##############################
# Comparison: v1 vs v2 vs v3 #
##############################

version_order = ['v1 (baseline)', 'v2 (log + threshold)', 'v3 (wider grids)']
missing = [v for v in version_order if v not in test_results]
if missing:
    print(f"Warning: no test results yet for {missing}. Run their evaluation cells first.\n")
available = [v for v in version_order if v in test_results]

df_test_summary = pd.concat([pd.DataFrame(test_results[v]) for v in available], ignore_index=True)
print("=========================================================================")
print("            TEST SET PERFORMANCE: v1 vs v2 vs v3 (all models)")
print("=========================================================================")
display(df_test_summary.round(4))

for metric, title in [
    ('Recall (Attack)', 'Test recall (Attack) by model and version:'),
    ('False alarms', 'Test false alarms (normal flows flagged as attacks) by model and version:'),
    ('F1-Score (Attack)', 'Test F1-Score (Attack) by model and version:'),
]:
    print(f"\n{title}")
    display(
        df_test_summary
        .pivot(index='Model', columns='Version', values=metric)
        [available]
        .round(4)
    )

In [ ]:
#############################################################
# Operating points of all versions at several recall levels #
#############################################################

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    roc_curve, roc_auc_score, confusion_matrix, precision_recall_fscore_support
)

# Same fitted models as in v1, v2 and v3; hyperparameters stay as selected. Only the threshold changes here:
# for each recall level it is picked on out-of-fold training scores, then applied once to the test set.
# Comparing the versions at the same recall levels shows what the tuning really bought.
# Reuses from earlier cells: fitted_searches, final_models, final_models_ext, cv_strategy,
# features/target train and test, and the plot style (MODEL_COLORS, SURFACE, INK, INK_2, MUTED,
# CM_CMAP, style_axes) from the v1 evaluation cell.
RECALL_LEVELS = [0.85, 0.90, 0.95, 0.98]
LEVEL_MARKERS = {0.85: 'o', 0.90: 's', 0.95: '^', 0.98: 'D'}
V1, V2, V3 = 'v1 (baseline)', 'v2 (log + threshold)', 'v3 (wider grids)'
VERSION_LINESTYLES = {V1: ':', V2: '--', V3: '-'}

fitted_pipelines = {
    V1: {name: search.best_estimator_ for name, search in fitted_searches.items()},
    V2: {name: tuned.estimator_ for name, tuned in final_models.items()},
    V3: {name: tuned.estimator_ for name, tuned in final_models_ext.items()},
}

def score_method(model):
    return 'predict_proba' if hasattr(model, 'predict_proba') else 'decision_function'

def positive_scores(model, X):
    method = score_method(model)
    scores = getattr(model, method)(X)
    return scores[:, 1] if method == 'predict_proba' else scores

operating_rows = []
test_scores_by = {}   # (version, model) -> scores on the test set
for version, pipes in fitted_pipelines.items():
    for name, pipe in pipes.items():
        method = score_method(pipe)
        oof = cross_val_predict(clone(pipe), features_train, target_train, cv=cv_strategy, method=method, n_jobs=-1)
        oof = oof[:, 1] if method == 'predict_proba' else oof
        _, tpr_train, thr_train = roc_curve(target_train, oof)
        scores = positive_scores(pipe, features_test)
        test_scores_by[(version, name)] = scores
        for level in RECALL_LEVELS:
            # Highest threshold that reaches the recall level on the out-of-fold training scores
            threshold = thr_train[np.argmax(tpr_train >= level)]
            preds = (scores >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(target_test, preds).ravel()
            prec, rec, f1, _ = precision_recall_fscore_support(
                target_test, preds, average='binary', pos_label=1
            )
            operating_rows.append({
                'Version': version,
                'Model': name,
                'Recall target': level,
                'Threshold': threshold,
                'Recall': rec,
                'Precision': prec,
                'Specificity': tn / (tn + fp),
                'False alarms': int(fp),
                'F1-Score': f1,
            })

df_operating = pd.DataFrame(operating_rows)
print("=========================================================================")
print("   ALL VERSIONS ON THE TEST SET AT SEVERAL RECALL LEVELS (ROC-AUC unchanged)")
print("=========================================================================")
display(df_operating.round(4))
print("Thresholds were picked on out-of-fold training scores, so the recall reached on the test set can differ slightly from the target.")

for column, title in [('False alarms', 'False alarms on the test set'), ('Precision', 'Precision')]:
    print(f"\n{title} by model, version and recall level:")
    display(
        df_operating
        .pivot_table(index=['Model', 'Version'], columns='Recall target', values=column, sort=False)
        .round(4)
    )

models_v3 = [name for (version, name) in test_scores_by if version == V3]
df_v3 = df_operating[df_operating['Version'] == V3]

# --- Figure 1: ROC curves of the v3 models with one marker per recall level ---
fig, ax = plt.subplots(figsize=(6.2, 5.4), facecolor=SURFACE)
style_axes(ax)
ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1, color=MUTED)
model_handles = []
for name in models_v3:
    scores = test_scores_by[(V3, name)]
    fpr, tpr, _ = roc_curve(target_test, scores)
    auc = roc_auc_score(target_test, scores)
    ax.plot(fpr, tpr, linewidth=2, color=MODEL_COLORS[name])
    model_handles.append(Line2D([0], [0], color=MODEL_COLORS[name], linewidth=2, label=f"{name} (AUC = {auc:.3f})"))
    for _, row in df_v3[df_v3['Model'] == name].iterrows():
        ax.scatter([1 - row['Specificity']], [row['Recall']], marker=LEVEL_MARKERS[row['Recall target']], s=80,
                   color=MODEL_COLORS[name], edgecolor=SURFACE, linewidth=1.5, zorder=3)
level_handles = [Line2D([0], [0], marker=m, color=MUTED, linestyle='', markersize=7, label=f"{lvl:.0%} recall level")
                 for lvl, m in LEVEL_MARKERS.items()]
first_legend = ax.legend(handles=model_handles, loc='lower right', frameon=False, labelcolor=INK, fontsize=9)
ax.add_artist(first_legend)
ax.legend(handles=level_handles, loc='center right', frameon=False, labelcolor=INK_2, fontsize=8.5)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel('False positive rate', color=INK_2)
ax.set_ylabel('True positive rate (recall)', color=INK_2)
ax.set_title('ROC curves on the test set with the recall levels marked (v3)', color=INK, loc='left', fontsize=11)
plt.tight_layout()
plt.show()

# --- Figure 2: the price of each recall level for the v3 models ---
fig, (ax_fa, ax_pr) = plt.subplots(1, 2, figsize=(9.6, 4.0), facecolor=SURFACE)
n_normal = int((target_test == 0).sum())
for ax, column, ylabel in [
    (ax_fa, 'False alarms', f'False alarms on the test set (of {n_normal} normal flows)'),
    (ax_pr, 'Precision', 'Precision'),
]:
    style_axes(ax)
    for name in models_v3:
        sub = df_v3[df_v3['Model'] == name]
        ax.plot(sub['Recall target'] * 100, sub[column], marker='o', markersize=7, linewidth=2,
                color=MODEL_COLORS[name], markeredgecolor=SURFACE, markeredgewidth=1.5, label=name)
    ax.set_xticks([lvl * 100 for lvl in RECALL_LEVELS])
    ax.set_xlabel('Recall level (%)', color=INK_2)
    ax.set_ylabel(ylabel, color=INK_2)
ax_fa.legend(loc='upper left', frameon=False, labelcolor=INK, fontsize=9)
ax_fa.set_title('False alarms rise with the recall level (v3)', color=INK, loc='left', fontsize=10.5)
ax_pr.set_title('Precision falls with the recall level (v3)', color=INK, loc='left', fontsize=10.5)
plt.tight_layout()
plt.show()

# --- Figure 3: does the tuning help at the same recall level? One column per model, one line per version ---
fig, axes = plt.subplots(2, len(models_v3), figsize=(3.6 * len(models_v3), 6.2), facecolor=SURFACE, sharex=True)
axes = np.atleast_2d(axes)
for col, name in enumerate(models_v3):
    for row_idx, (column, ylabel) in enumerate([('False alarms', 'False alarms (test set)'), ('Precision', 'Precision')]):
        ax = axes[row_idx, col]
        style_axes(ax)
        for version in (V1, V2, V3):
            sub = df_operating[(df_operating['Model'] == name) & (df_operating['Version'] == version)]
            ax.plot(sub['Recall target'] * 100, sub[column], linestyle=VERSION_LINESTYLES[version], marker='o', markersize=6,
                    linewidth=2, color=MODEL_COLORS[name], markeredgecolor=SURFACE, markeredgewidth=1.2)
        if col == 0:
            ax.set_ylabel(ylabel, color=INK_2)
        if row_idx == 0:
            ax.set_title(name, color=INK, fontsize=10.5)
        if row_idx == 1:
            ax.set_ylim(0, 1.02)
            ax.set_xticks([lvl * 100 for lvl in RECALL_LEVELS])
            ax.set_xlabel('Recall level (%)', color=INK_2)
version_handles = [Line2D([0], [0], color=MUTED, linewidth=2, linestyle=VERSION_LINESTYLES[v], label=v) for v in (V1, V2, V3)]
fig.legend(handles=version_handles, loc='upper right', ncol=3, frameon=False, labelcolor=INK_2, fontsize=8.5)
fig.suptitle('Same recall levels, all versions (each model has its own false-alarm scale)',
             color=INK, x=0.01, ha='left', fontsize=10.5)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

# --- Figure 4: confusion matrices of the v3 models, one row per recall level ---
fig, axes = plt.subplots(len(RECALL_LEVELS), len(models_v3),
                         figsize=(3.2 * len(models_v3), 2.7 * len(RECALL_LEVELS)), facecolor=SURFACE)
for r_idx, level in enumerate(RECALL_LEVELS):
    for c_idx, name in enumerate(models_v3):
        ax = axes[r_idx, c_idx]
        threshold = df_v3[(df_v3['Model'] == name) & (df_v3['Recall target'] == level)]['Threshold'].iloc[0]
        cm = confusion_matrix(target_test, (test_scores_by[(V3, name)] >= threshold).astype(int))
        share = cm / cm.sum(axis=1, keepdims=True)
        ax.imshow(share, cmap=CM_CMAP, vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i, j]}\n({share[i, j]:.0%})", ha='center', va='center',
                        color='white' if share[i, j] > 0.5 else INK, fontsize=9)
        ax.set_xticks([0, 1], ['Normal', 'Attack'], color=INK_2, fontsize=8)
        ax.set_yticks([0, 1], ['Normal', 'Attack'], color=INK_2, fontsize=8)
        ax.set_title(f"{name}, {level:.0%} recall level", color=INK, fontsize=9.5)
        if c_idx == 0:
            ax.set_ylabel('Actual', color=INK_2)
        if r_idx == len(RECALL_LEVELS) - 1:
            ax.set_xlabel('Predicted', color=INK_2)
        for spine in ax.spines.values():
            spine.set_visible(False)
fig.suptitle('Confusion matrices on the test set by recall level (v3, colour = share of each actual class)',
             color=INK, x=0.01, ha='left', fontsize=10.5)
plt.tight_layout()
plt.show()